# RAG pipeline notebook

## Part 1: upload validation, text extraction and chunking

This notebook runs the real ingestion code from `app/ingestion/` step by step on the documents in `corpus/`:

1. **Validation**: is the upload safe and really a PDF or DOCX?
2. **Extraction**: turn each file into citable sections (PDF pages, DOCX page labels and headings).
3. **Chunking**: split sections into overlapping passages with configurations **A** (1000/150) and **B** (500/75).

Only the corpus documents are used. The assignment brief and the project plan are not part of the corpus. No API keys are needed, and the settings are created with `_env_file=None`, so `.env` is never read.

Later parts (embedding, retrieval, answering) will be added to this notebook as those steps are built.

In [1]:
import statistics
import sys
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from app.config import CHUNK_CONFIGS, CORPUS_DIR, Settings
from app.errors import AppError
from app.ingestion.chunk import chunk_document
from app.ingestion.extract import extract_text, normalize_lines
from app.ingestion.validation import sanitize_filename, validate_upload

settings = Settings(_env_file=None)  # no .env: this notebook needs no secrets
corpus_files = sorted(p for p in CORPUS_DIR.iterdir() if p.suffix.lower() in ('.pdf', '.docx'))
for path in corpus_files:
    print(f'{path.name:45} {path.stat().st_size / 1024:8.0f} KB')

NIST.AI.100-1.pdf                                 1901 KB
NIST.AI.600-1.docx                                  87 KB
OWASP-Top-10-LLM-Applications-2025.docx             67 KB


## 1. Validation

`validate_upload` sanitises the file name, checks the extension, size and file signature (the first bytes of the file), and for DOCX checks the ZIP structure and its uncompressed size (ZIP-bomb guard). The SHA-256 hash of the bytes becomes the duplicate check, and its first 16 characters the document ID.

In [2]:
validated = {}
for path in corpus_files:
    v = validate_upload(path.name, path.read_bytes(), settings)
    validated[path.name] = v
    print(f'{v.filename:45} type={v.file_type:4}  size={v.size_bytes:>9,} B  '
          f'sha256={v.sha256[:12]}...  document_id={v.document_id}')

NIST.AI.100-1.pdf                             type=pdf   size=1,946,127 B  sha256=7576edb531d9...  document_id=7576edb531d98488
NIST.AI.600-1.docx                            type=docx  size=   89,297 B  sha256=863ec848e54e...  document_id=863ec848e54e79bc
OWASP-Top-10-LLM-Applications-2025.docx       type=docx  size=   68,108 B  sha256=6e711bdf3b60...  document_id=6e711bdf3b6053db


Uploads that must be rejected, with the HTTP status code the API will return:

In [3]:
bad_uploads = {
    'notes.txt': b'plain text',
    'renamed.pdf': b'This is a text file renamed to .pdf',
    'empty.docx': b'',
    'broken.docx': b'PK\x03\x04 looks like a ZIP but is not one',
    'old-word.docx': b'\xd0\xcf\x11\xe0' + b'\x00' * 64,
}
for name, data in bad_uploads.items():
    try:
        validate_upload(name, data, settings)
        print(f'{name:15} -> accepted (unexpected!)')
    except AppError as error:
        print(f'{name:15} -> {error.status_code} {error.code}: {error.message}')

print('\nFile names are reduced to a safe base name:')
for raw in ['../../etc/report.pdf', 'C:\\Users\\me\\notes.docx', 'bad\x00name.pdf']:
    print(f'  {raw!r:32} -> {sanitize_filename(raw)!r}')

notes.txt       -> 415 unsupported_file_type: This file type is not supported. Upload a PDF or DOCX file.
renamed.pdf     -> 415 unsupported_file_type: The file has a .pdf extension but is not a PDF.
empty.docx      -> 400 empty_document: The uploaded file is empty.
broken.docx     -> 400 invalid_document: The DOCX file is corrupt.
old-word.docx   -> 415 unsupported_file_type: This looks like a legacy .doc or password-protected Word file. Save it as a normal .docx and retry.

File names are reduced to a safe base name:
  '../../etc/report.pdf'           -> 'report.pdf'
  'C:\\Users\\me\\notes.docx'      -> 'notes.docx'
  'bad\x00name.pdf'                -> 'badname.pdf'


## 2. Extraction

* **PDF**: one section per page (page numbers from 1). Running headers and footers that repeat on most pages are removed. Each page also gets a section path from the PDF's bookmarks, **but only bookmarks whose heading really appears on the page are used**: several bookmarks in NIST AI 100-1 point to the wrong page.
* **DOCX**: the corpus DOCX files were converted from PDFs and mark each original page with a `Heading 1` paragraph such as `Page 12`. These set the page number. Ordinary headings (if a document has them) build the section path.

In [4]:
extracted = {}
for name, v in validated.items():
    started = time.perf_counter()
    doc = extract_text(v, settings)
    extracted[name] = doc
    print(f'{name}\n  pages={doc.page_count}  sections={len(doc.sections)}  empty pages={doc.empty_pages}  '
          f'characters={doc.char_count:,}  time={time.perf_counter() - started:.2f}s')
    print(f'  removed header/footer lines: {doc.removed_boilerplate}')

NIST.AI.100-1.pdf
  pages=48  sections=48  empty pages=[]  characters=104,555  time=2.00s
  removed header/footer lines: ['NIST AI 100-1 AI RMF 1.0', 'Page 1']


NIST.AI.600-1.docx
  pages=64  sections=65  empty pages=[]  characters=158,100  time=0.42s
  removed header/footer lines: ['1']


OWASP-Top-10-LLM-Applications-2025.docx
  pages=45  sections=46  empty pages=[]  characters=90,340  time=0.26s
  removed header/footer lines: ['1genai.owasp.org', 'OWASP Top 10 for LLM Applications v2.0']


Header and footer removal on one PDF page: raw `pypdf` text compared with the cleaned section.

In [5]:
from pypdf import PdfReader

pdf_name = 'NIST.AI.100-1.pdf'
raw_lines = normalize_lines(PdfReader(CORPUS_DIR / pdf_name).pages[9].extract_text())
clean = next(s for s in extracted[pdf_name].sections if s.page == 10)
print('RAW first line :', raw_lines[0])
print('RAW last line  :', raw_lines[-1])
print('CLEAN first line:', clean.text.splitlines()[0])
print('CLEAN last line :', clean.text.splitlines()[-1])
print('Section path   :', clean.section)

RAW first line : NIST AI 100-1 AI RMF 1.0
RAW last line  : Page 5
CLEAN first line: Fig. 1. Examples of potential harms related to AI systems. Trustworthy AI systems and their
CLEAN last line : identifying and tracking emergent risks and considering techniques for measuring them.
Section path   : Framing Risk > Challenges for AI Risk Management > Risk Measurement


Section path per PDF page, from verified bookmarks (pages before the first bookmark have none):

In [6]:
previous = object()
for s in extracted[pdf_name].sections:
    if s.section != previous:
        print(f'from page {s.page:>2}: {s.section}')
        previous = s.section

from page  1: None
from page  6: Executive Summary
from page  9: Framing Risk > Understanding and Addressing Risks, Impacts, and Harms
from page 10: Framing Risk > Challenges for AI Risk Management > Risk Measurement
from page 12: Framing Risk > Challenges for AI Risk Management > Risk Prioritization
from page 13: Framing Risk > Challenges for AI Risk Management > Organizational Integration and Management of Risk
from page 14: Audience
from page 17: AI Risks and Trustworthiness
from page 18: AI Risks and Trustworthiness > Valid and Reliable
from page 19: AI Risks and Trustworthiness > Safe
from page 20: AI Risks and Trustworthiness > Accountable and Transparent
from page 21: AI Risks and Trustworthiness > Explainable and Interpretable
from page 22: AI Risks and Trustworthiness > Fair – with Harmful Bias Managed
from page 24: Effectiveness of the AI RMF
from page 25: AI RMF Core
from page 26: AI RMF Core > Govern
from page 29: AI RMF Core > Map
from page 33: AI RMF Core > Measure
from p

DOCX page labels: each section knows its page in the original PDF.

In [7]:
owasp = extracted['OWASP-Top-10-LLM-Applications-2025.docx']
for s in owasp.sections[:6]:
    preview = s.text[:90].replace('\n', ' / ')
    print(f'order={s.order:<2} page={s.page!s:<4} section={s.section!s:<5} | {preview}')

order=0  page=None section=None  | OWASP Top 10 for LLM Applications 2025 / Converted from the official OWASP release PDF for t
order=1  page=1    section=None  | OWASP PDF v4.2.0a 20241114-202703 / OWASP Top 10 for / LLM Applications 2025 / Version 2025 / Nove
order=2  page=2    section=None  | LICENSE AND USAGE / This document is licensed under Creative Commons, CC BY-SA 4.0. / You are 
order=3  page=3    section=None  | Table of Contents / Letter from the Project Leads . . . . . . . . . . . . . . . . . . . . . 
order=4  page=4    section=None  | Common Examples of Vulnerability . . . . . . . . . . . . . . . . . . . . . . . . . . . . .
order=5  page=5    section=None  | Letter from the Project Leads / The OWASP Top 10 for Large Language Model Applications start


## 3. Chunking

Chunks never cross a section, so each one has exactly one page to cite. LangChain's `RecursiveCharacterTextSplitter` picks the cut points (blank line, line, sentence, word); our code then adds the overlap by extending each chunk back to a word boundary. We do this because the splitter's own overlap only repeats whole lines, which on this corpus gave config B almost no real overlap.

In [8]:
chunked = {}
print(f"{'document':45} cfg  chunks  min/mean/max length   mean overlap")
for name, v in validated.items():
    for cfg_name, cfg in CHUNK_CONFIGS.items():
        chunks = chunk_document(v, extracted[name], cfg, settings.min_chunk_chars)
        chunked[(name, cfg_name)] = chunks
        lengths = [len(c.text) for c in chunks]
        overlaps = [p.char_end - c.char_start for p, c in zip(chunks, chunks[1:])
                    if p.section_order == c.section_order]
        print(f'{name:45} {cfg_name:3} {len(chunks):6}  {min(lengths):4}/{statistics.mean(lengths):4.0f}/{max(lengths):4}'
              f'        {statistics.mean(overlaps):5.0f} chars (target {cfg.chunk_overlap})')

document                                      cfg  chunks  min/mean/max length   mean overlap
NIST.AI.100-1.pdf                             A      152    76/ 788/ 998          146 chars (target 150)
NIST.AI.100-1.pdf                             B      304    76/ 403/ 499           70 chars (target 75)
NIST.AI.600-1.docx                            A      230   162/ 790/ 998          144 chars (target 150)
NIST.AI.600-1.docx                            B      439   126/ 419/ 499           69 chars (target 75)


OWASP-Top-10-LLM-Applications-2025.docx       A      129    76/ 794/ 994          145 chars (target 150)
OWASP-Top-10-LLM-Applications-2025.docx       B      257    76/ 408/ 499           69 chars (target 75)


One chunk with the metadata that will be stored with its vector and used for citations:

In [9]:
sample = chunked[(pdf_name, 'A')][20]
for key, value in sample.metadata().items():
    print(f'{key:14} {value}')
print('\ntext:\n' + sample.text)

chunk_id       7576edb531d98488:20
document_id    7576edb531d98488
filename       NIST.AI.100-1.pdf
file_type      pdf
chunk_index    20
page           9
section        Framing Risk > Understanding and Addressing Risks, Impacts, and Harms
section_order  8
char_start     0
char_end       794
sha256         7576edb531d9848825814ee88e28b1795d3a84b435b4b797d3670eafdc4a89f1
chunk_config   A

text:
Part 1: Foundational Information
1. Framing Risk
AI risk management offers a path to minimize potential negative impacts of AI systems,
such as threats to civil liberties and rights, while also providing opportunities to maximize
positive impacts. Addressing, documenting, and managing AI risks and potential negative
impacts effectively can lead to more trustworthy AI systems.
1.1 Understanding and Addressing Risks, Impacts, and Harms
In the context of the AI RMF, risk refers to the composite measure of an event’s probability
of occurring and the magnitude or degree of the consequences of the corre

Overlap between two neighbouring chunks (config B): the end of one chunk is repeated at the start of the next.

In [10]:
chunks_b = chunked[(pdf_name, 'B')]
first, second = next((a, b) for a, b in zip(chunks_b, chunks_b[1:])
                     if a.section_order == b.section_order and a.page == 10)
shared = first.text[second.char_start - first.char_start:]
print(f'chunk {first.chunk_index} ends with:   ...{first.text[-110:]!r}')
print(f'chunk {second.chunk_index} starts with: {second.text[:110]!r}...')
print(f'shared text ({len(shared)} chars): {shared!r}')

chunk 47 ends with:   ...'They should be taken into account when managing\nrisks in pursuit of AI trustworthiness.\n1.2.1 Risk Measurement'
chunk 48 starts with: 'managing\nrisks in pursuit of AI trustworthiness.\n1.2.1 Risk Measurement\nAI risks or failures that are not well'...
shared text (71 chars): 'managing\nrisks in pursuit of AI trustworthiness.\n1.2.1 Risk Measurement'


Automatic checks over every chunk of every document, for both configurations:

In [11]:
total = 0
for (name, cfg_name), chunks in chunked.items():
    cfg = CHUNK_CONFIGS[cfg_name]
    sections = {s.order: s for s in extracted[name].sections}
    assert len({c.chunk_id for c in chunks}) == len(chunks), 'duplicate chunk IDs'
    for c in chunks:
        assert len(c.text) <= cfg.chunk_size, 'chunk too long'
        assert sections[c.section_order].text[c.char_start:c.char_end] == c.text, 'offsets wrong'
        assert c.document_id and c.filename == name
        assert c.page is not None or c.section_order == 0, 'missing page'
    total += len(chunks)
print(f'All checks passed for {total} chunks across {len(chunked)} document/config pairs.')

All checks passed for 1511 chunks across 6 document/config pairs.


## 4. Observations (from the run above)

**Validation**
* All three corpus files pass. The five bad uploads are rejected with the status codes the API will use: 415 for an unsupported extension, a renamed text file or a legacy/password-protected Word file; 400 for an empty or corrupt file.

**Extraction**
* NIST AI 100-1 (PDF): 48 pages, none empty. The running header `NIST AI 100-1 AI RMF 1.0` and the `Page N` footer are removed from every page (`Page 1` is the example shown for that pattern).
* The PDF's bookmarks are imprecise, so only bookmarks whose heading appears on the page are used. Pages 1–5 (cover, notices, contents) get no section, and the appendices now start on their real pages (40, 43, 45, 47).
* **Limitation:** sections are per page. When two sections start on the same page, the later one names the whole page. For example, page 10 is labelled *Risk Measurement* even though it starts with the end of section 1.1, and *Risk Tolerance* (page 12), *Secure and Resilient* (page 20) and *Privacy-Enhanced* (page 22) never appear as labels.
* The two DOCX files carry page labels from their PDF conversion, so their chunks cite a page. They have no content headings, so `section` is empty for them. Their first section (order 0) is the conversion note, which has no page.
* The OWASP DOCX keeps its table of contents (pages 3–4) as ordinary text. Lines full of dot leaders could match many questions at retrieval time; check this in the retrieval evaluation.

**Chunking**
* Config A gives 511 chunks in total and config B gives 1,000. Every chunk is within its size limit and is an exact slice of its section (1,511 chunks checked).
* Measured overlap is close to the target: about 145 characters for A (target 150) and about 70 for B (target 75). It is slightly below target because overlap starts at a word boundary.
* Which configuration retrieves better is **not decided here**. That is measured in the retrieval evaluation (hit rate@k, recall@k, MRR) and reported in EVAL.md.
